In [1]:
import torch
import lightning.pytorch as ptl
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

import boda

/proj/bmfm/users/sanjoy/miniforge3/envs/malinois_python311/lib/python3.11/site-packages/lightning/fabric/__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


# Set up

## Get data

In [2]:
!gsutil cp gs://tewhey-public-data/CODA_resources/Table_S2__MPRA_dataset.txt ./

Copying gs://tewhey-public-data/CODA_resources/Table_S2__MPRA_dataset.txt...
| [1 files][267.2 MiB/267.2 MiB]                                                
Operation completed over 1 objects/267.2 MiB.                                    


## Pick modules
Pick modules to define:
1. The data, how it's preprocessed and train/val/test split
2. The model, the architecture setup, loss function, etc.
3. The graph, how the data is used to train the model (i.e. training loop)

In [2]:
data_module = boda.data.MPRA_DataModule
model_module= boda.model.BassetBranched
graph_module= boda.graph.CNNBasicTraining

## Initalize Data
I added chr1 to test and chr2 to val to speed up this example. I also removed the reverse complementation data augmentation.

In [4]:
data = data_module(
    datafile_path="/proj/bmfm/datasets/omics/genome/finetune_datasets/snv_mpra_Tewhey_v3_hg37/split_Gosai_minus_mpratest/K562_biallelic_200/train.csv", 
    sep='\t', sequence_column='snpified_seq',
    synth_val_pct=0.0, synth_test_pct=99.98,
    val_chrs=['2','19','21','X'], test_chrs=['1','7','13'], 
    activity_columns=['K562_log2FC'],
    batch_size=1024, padded_seq_len=600, 
    use_reverse_complements=False, 
    duplication_cutoff=2.0, 
    num_workers=8
)


# Modified by Sanjoy

In [3]:
import os
import torch
import pandas as pd
import lightning.pytorch as pl
from functools import partial
from torch.utils.data import DataLoader, TensorDataset, Dataset
from boda.common import constants, utils
from boda.data.mpra_datamodule import DNAActivityDataset   # reuse the existing Dataset class

class SimpleMPRA_DataModule(pl.LightningDataModule):
    """
    Simplified MPRA DataModule that accepts pre-split train / val / test files.

    Args:
        datafile_path   (str)        : Path to the TRAIN file.
        val_chrs        (str|list)   : If str  → path to a val  file.
                                       If list → chr list to filter from datafile_path (needs chr_column).
        test_chrs       (str|list)   : If str  → path to a test file.
                                       If list → chr list to filter from datafile_path (needs chr_column).
        sequence_column (str)        : Column name for DNA sequences.
        activity_columns(list[str])  : Column names for activity values.
        chr_column      (str)        : Column name for chromosomes (only needed when val/test_chrs are lists).
        sep             (str)        : File delimiter.
        batch_size      (int)        : Mini-batch size.
        padded_seq_len  (int)        : Total sequence length after padding.
        left_flank      (str)        : Upstream padding sequence.
        right_flank     (str)        : Downstream padding sequence.
        num_workers     (int)        : DataLoader worker count.
        duplication_cutoff (float)   : Max-activity cutoff for oversampling high-signal seqs in train.
        use_reverse_complements(bool): Augment training set with reverse complements.
    """

    def __init__(self,
                 datafile_path,
                 val_chrs=None,
                 test_chrs=None,
                 sequence_column='sequence',
                 activity_columns=['HepG2_log2FC', 'SKNSH_log2FC'],
                 chr_column='chr',
                 sep='\t',
                 batch_size=32,
                 padded_seq_len=600,
                 left_flank=constants.MPRA_UPSTREAM,
                 right_flank=constants.MPRA_DOWNSTREAM,
                 num_workers=8,
                 duplication_cutoff=None,
                 use_reverse_complements=False,
                 **kwargs):
        super().__init__()
        self.datafile_path          = datafile_path
        self.val_chrs               = val_chrs
        self.test_chrs              = test_chrs
        self.sequence_column        = sequence_column
        self.activity_columns       = activity_columns
        self.chr_column             = chr_column
        self.sep                    = sep
        self.batch_size             = batch_size
        self.padded_seq_len         = padded_seq_len
        self.left_flank             = left_flank
        self.right_flank            = right_flank
        self.num_workers            = num_workers
        self.duplication_cutoff     = duplication_cutoff
        self.use_reverse_complements= use_reverse_complements

        self.pad_column_name = 'padded_seq'
        self.padding_fn = partial(utils.row_pad_sequence,
                                  in_column_name=self.sequence_column,
                                  padded_seq_len=self.padded_seq_len,
                                  upStreamSeq=self.left_flank,
                                  downStreamSeq=self.right_flank)

        self.dataset_train = None
        self.dataset_val   = None
        self.dataset_test  = None

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _load_columns(self, val_chrs=None, test_chrs=None):
        """Return the column list to parse, adding chr_column only when needed."""
        cols = [self.sequence_column, *self.activity_columns]
        need_chr = (isinstance(val_chrs, list) and val_chrs) or \
                   (isinstance(test_chrs, list) and test_chrs)
        if need_chr:
            cols.append(self.chr_column)
        return cols

    def _df_to_dataset(self, df, is_train=False):
        """
        Convert a DataFrame (already filtered/loaded) into a DNAActivityDataset.
        Pads sequences, one-hot encodes, stacks into tensors.
        """
        df = df.copy()
        print(f'  Padding {len(df):,} sequences...')
        df[self.pad_column_name] = df.apply(self.padding_fn, axis=1)

        list_tensor_seq = []
        for _, row in df.iterrows():
            list_tensor_seq.append(utils.row_dna2tensor(row, in_column_name=self.pad_column_name))

        sequences  = torch.stack(list_tensor_seq)
        activities = torch.Tensor(df[self.activity_columns].to_numpy())

        if is_train:
            return DNAActivityDataset(
                sequences, activities,
                sort_tensor=torch.max(activities, dim=-1).values,
                duplication_cutoff=self.duplication_cutoff,
                use_reverse_complements=self.use_reverse_complements
            )
        else:
            return TensorDataset(sequences, activities)

    def _load_split(self, source, train_df=None, label=''):
        """
        Load a split from either:
          - a file path (str)  → read directly with the same columns as train
          - a chr list (list)  → filter train_df by those chromosomes
        Returns a raw DataFrame ready for _df_to_dataset().
        """
        if isinstance(source, str):                        # ← file path
            print(f'  Loading {label} from file: {source}')
            cols = [self.sequence_column, *self.activity_columns]
            return utils.parse_file(file_path=source, columns=cols, sep=self.sep)
        elif isinstance(source, list) and source:          # ← chr list
            print(f'  Filtering {label} from train file by chromosomes: {source}')
            assert train_df is not None and self.chr_column in train_df.columns, \
                "chr_column must be present in train_df when val/test_chrs is a list"
            return train_df[train_df[self.chr_column].isin(set(source))].reset_index(drop=True)
        else:
            return None

    # ------------------------------------------------------------------
    # setup — called by Lightning before fit/test
    # ------------------------------------------------------------------

    def setup(self, stage='train'):
        print('-' * 50)
        print('Setting up SimpleMPRA_DataModule...\n')

        # 1. Load train file — only the columns we need
        cols = self._load_columns(self.val_chrs, self.test_chrs)
        print(f'Loading train file: {self.datafile_path}')
        train_df = utils.parse_file(file_path=self.datafile_path, columns=cols, sep=self.sep)
        print(f'  {len(train_df):,} rows loaded.\n')

        # 2. Build train dataset (entire train file, no chr filtering)
        print('Building TRAIN dataset:')
        self.dataset_train = self._df_to_dataset(train_df, is_train=True)
        print(f'  → {len(self.dataset_train):,} examples (incl. augmentation)\n')

        # 3. Build val dataset
        val_df = self._load_split(self.val_chrs, label='VAL')
        if val_df is not None:
            print('Building VAL dataset:')
            self.dataset_val = self._df_to_dataset(val_df, is_train=False)
            print(f'  → {len(self.dataset_val):,} examples\n')

        # 4. Build test dataset
        test_df = self._load_split(self.test_chrs, label='TEST')
        if test_df is not None:
            print('Building TEST dataset:')
            self.dataset_test = self._df_to_dataset(test_df, is_train=False)
            print(f'  → {len(self.dataset_test):,} examples\n')

        print('-' * 50)

    # ------------------------------------------------------------------
    # DataLoaders
    # ------------------------------------------------------------------

    def train_dataloader(self):
        return DataLoader(self.dataset_train, batch_size=self.batch_size,
                          shuffle=True, num_workers=self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.dataset_val, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers)

    def test_dataloader(self):
        return DataLoader(self.dataset_test, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers)

In [ ]:
# Cell 8 — initialize data with pre-split files
data = SimpleMPRA_DataModule(
    datafile_path  = "/proj/bmfm/datasets/omics/genome/finetune_datasets/snv_mpra_Tewhey/split_Gosai_minus_mpratest/K562_biallelic_200/train.csv",          # ← your train file
    val_chrs       = "/proj/bmfm/datasets/omics/genome/finetune_datasets/snv_mpra_Tewhey/split_Gosai_minus_mpratest/K562_biallelic_200/dev.csv",            # ← string = file path
    test_chrs      = "/proj/bmfm/datasets/omics/genome/finetune_datasets/snv_mpra_Tewhey/split_Gosai_and_mpra/K562/test.csv"  #"/proj/bmfm/datasets/omics/genome/finetune_datasets/snv_mpra_Tewhey/split_Gosai_minus_mpratest/K562_biallelic_200/test.csv",           # ← string = file path
    sep            = ',',
    activity_columns=['K562_log2FC'],
    batch_size     = 1024,
    padded_seq_len = 600,
    use_reverse_complements=False,
    duplication_cutoff=2.0,
    num_workers    = 8
)

In [5]:
# Call setup so the dataloader is ready to use
data.setup()

# Get the train dataloader
train_loader = data.train_dataloader()

# Grab one batch
batch = next(iter(train_loader))
sequences, labels = batch

print("=== Batch shapes ===")
print(f"Sequences : {sequences.shape}  → (batch_size, 4 nucleotides, padded_seq_len)")
print(f"Labels    : {labels.shape}    → (batch_size, n_activity_columns)")

print("\n=== First 3 sequences (one-hot encoded, first 10 positions) ===")
for i in range(3):
    print(f"  Sample {i}: {sequences[i, :, :10]}")

print("\n=== First 3 labels ===")
for i in range(3):
    print(f"  Sample {i}: K562_log2FC={labels[i,0]:.4f}")

print(f"\n=== Dataloader stats ===")
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(data.val_dataloader())}")
print(f"Test batches  : {len(data.test_dataloader())}")

--------------------------------------------------
Setting up SimpleMPRA_DataModule...

Loading train file: /proj/bmfm/datasets/omics/genome/finetune_datasets/snv_mpra_Tewhey/split_Gosai_minus_mpratest/K562_biallelic_200/train.csv


  598,477 rows loaded.

Building TRAIN dataset:
  Padding 598,477 sequences...


/tmp/ipykernel_1823255/437675543.py:103: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  activities = torch.Tensor(df[self.activity_columns].to_numpy())


  → 653,237 examples (incl. augmentation)

  Loading VAL from file: /proj/bmfm/datasets/omics/genome/finetune_datasets/snv_mpra_Tewhey/split_Gosai_minus_mpratest/K562_biallelic_200/dev.csv
Building VAL dataset:
  Padding 55,961 sequences...
  → 55,961 examples

  Loading TEST from file: /proj/bmfm/datasets/omics/genome/finetune_datasets/snv_mpra_Tewhey/split_Gosai_minus_mpratest/K562_biallelic_200/test.csv
Building TEST dataset:
  Padding 60,032 sequences...
  → 60,032 examples

--------------------------------------------------
=== Batch shapes ===
Sequences : torch.Size([1024, 4, 600])  → (batch_size, 4 nucleotides, padded_seq_len)
Labels    : torch.Size([1024, 1])    → (batch_size, n_activity_columns)

=== First 3 sequences (one-hot encoded, first 10 positions) ===
  Sample 0: tensor([[0., 0., 1., 0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 1., 1., 1., 0., 1., 1.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.]])
  Sample

## Initialize Model

In [6]:
model = model_module(
    n_outputs=1, 
    n_linear_layers=1, linear_channels=1000,
    linear_activation='ReLU', linear_dropout_p=0.12, 
    n_branched_layers=3, branched_channels=140, 
    branched_activation='ReLU', branched_dropout_p=0.56, 
    loss_criterion='L1KLmixed', loss_args={'beta':5.0}
)

## Append Graph to Model
Augment the model class to append functions from the graph module. A downside to this structure is that you need to make sure all relevent Graph args are defined (even if None is an acceptable default). This is because the `__init__` block in the Graph class doesn't run.

In [7]:
graph_args = {
    'optimizer': 'Adam', 
    'optimizer_args': {
        'lr': 0.0033, 'betas':[0.9, 0.999], 
        'weight_decay': 3.43e-4, 'amsgrad': True
    },
    'scheduler': 'CosineAnnealingWarmRestarts', 
    'scheduler_monitor': None, 
    'scheduler_interval': 'step',
    'scheduler_args': {
        'T_0': 4096,
    }
}

graph = graph_module(
    model = model,
    **graph_args
)

In [8]:
model(torch.randn(10,4,600))

tensor([[-0.0802],
        [-0.0802],
        [-0.0804],
        [-0.0804],
        [-0.0802],
        [-0.0801],
        [-0.0802],
        [-0.0805],
        [-0.0804],
        [-0.0802]], grad_fn=<PermuteBackward0>)

## Lightning trainer
Normally we train for more epochs, but reduced in this example. Update `min_epochs` and `max_epochs` accordingly.

In [9]:
checkpoint_callback = ModelCheckpoint(
    save_top_k=1, 
    monitor='prediction_mean_spearman', 
    mode='max'
)

stopping_callback = EarlyStopping(
    monitor='prediction_mean_spearman', 
    patience=5,
    mode='max'
)

trainer = ptl.Trainer(
    accelerator='cpu', devices=1, 
    min_epochs=2, max_epochs=5, # <- we use min_epochs=60, max_epochs=200
    precision=16, callbacks= [
        checkpoint_callback,
        stopping_callback
    ]
)

/proj/bmfm/users/sanjoy/miniforge3/envs/malinois_python311/lib/python3.11/site-packages/lightning/fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
/proj/bmfm/users/sanjoy/miniforge3/envs/malinois_python311/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/accelerator_connector.py:512: You passed `Trainer(accelerator='cpu', precision='16-mixed')` but AMP with fp16 is not supported on CPU. Using `precision='bf16-mixed'` instead.
Using bfloat16 Automatic Mixed Precision (AMP)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/proj/bmfm/users/sanjoy/miniforge3/envs/malinois_python311/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


## Train model

In [10]:
import torch
from boda.graph.cnn_prediction import CNNBasicTraining
from boda.graph.utils import spearman_correlation, shannon_entropy

# ── patch: accumulate outputs in validation_step ──────────────────────────────
_orig_validation_step = CNNBasicTraining.validation_step

def _patched_validation_step(self, batch, batch_idx):
    out = _orig_validation_step(self, batch, batch_idx)
    if not hasattr(self, '_val_outputs'):
        self._val_outputs = []
    self._val_outputs.append(out)
    return out

CNNBasicTraining.validation_step = _patched_validation_step

# ── patch: rename hook and read from self._val_outputs ────────────────────────
def _patched_on_validation_epoch_end(self):
    val_step_outputs = getattr(self, '_val_outputs', [])
    if not val_step_outputs:
        return

    arit_mean = torch.stack([b['loss']   for b in val_step_outputs], dim=0).mean()
    harm_mean = torch.stack([b['metric'] for b in val_step_outputs], dim=0) \
                    .mean(dim=0).pow(-1).mean().pow(-1)
    epoch_preds  = torch.cat([b['preds']  for b in val_step_outputs], dim=0)
    epoch_labels = torch.cat([b['labels'] for b in val_step_outputs], dim=0)

    spearman, mean_spearman = spearman_correlation(epoch_preds, epoch_labels)
    shannon_pred  = shannon_entropy(epoch_preds)
    shannon_label = shannon_entropy(epoch_labels)
    _, specificity_mean_spearman = spearman_correlation(shannon_pred, shannon_label)

    self.aug_log(external_metrics={
        'current_epoch':            self.current_epoch,
        'arithmetic_mean_loss':     arit_mean,
        'harmonic_mean_loss':       harm_mean,
        'prediction_mean_spearman': mean_spearman.item(),
        'entropy_spearman':         specificity_mean_spearman.item(),
    })
    self._val_outputs.clear()   # free memory each epoch

# Remove the old hook (which Lightning v2 rejects) and install the new one
if hasattr(CNNBasicTraining, 'validation_epoch_end'):
    del CNNBasicTraining.validation_epoch_end
CNNBasicTraining.on_validation_epoch_end = _patched_on_validation_epoch_end

In [ ]:
trainer.fit(graph, data)

--------------------------------------------------
Setting up SimpleMPRA_DataModule...

Loading train file: /proj/bmfm/datasets/omics/genome/finetune_datasets/snv_mpra_Tewhey/split_Gosai_minus_mpratest/K562_biallelic_200/train.csv
  598,477 rows loaded.

Building TRAIN dataset:
  Padding 598,477 sequences...
  → 653,237 examples (incl. augmentation)

  Loading VAL from file: /proj/bmfm/datasets/omics/genome/finetune_datasets/snv_mpra_Tewhey/split_Gosai_minus_mpratest/K562_biallelic_200/dev.csv
Building VAL dataset:
  Padding 55,961 sequences...
  → 55,961 examples

  Loading TEST from file: /proj/bmfm/datasets/omics/genome/finetune_datasets/snv_mpra_Tewhey/split_Gosai_minus_mpratest/K562_biallelic_200/test.csv
Building TEST dataset:
  Padding 60,032 sequences...



  | Name      | Type           | Params | Mode 
-----------------------------------------------------
0 | model     | BassetBranched | 3.7 M  | train
1 | criterion | L1KLmixed      | 0      | train
-----------------------------------------------------
3.7 M     Trainable params
0         Non-trainable params
3.7 M     Total params
14.991    Total estimated model params size (MB)


  → 60,032 examples

--------------------------------------------------
Found 3747661 parameters


Sanity Checking: |                                                                                            …

/proj/bmfm/users/sanjoy/miniforge3/envs/malinois_python311/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/proj/bmfm/users/sanjoy/miniforge3/envs/malinois_python311/lib/python3.11/site-packages/torch/nn/modules/loss.py:558: UserWarning: reduction: 'mean' divides the total loss by both the batch size and the support size.'batchmean' divides only by the batch size, and aligns with the KL div math definition.'mean' will be changed to behave the same as 'batchmean' in the next major release.
  return F.kl_div(


...


--------------------------------------------------------------------------------------------------------------------------------------------------------
| current_epoch: 0.00000 | arithmetic_mean_loss: 0.11776 | harmonic_mean_loss: 1.26759 | prediction_mean_spearman: 0.52128 | entropy_spearman: 1.00000 |
--------------------------------------------------------------------------------------------------------------------------------------------------------



Validation: |                                                                                                 …


--------------------------------------------------------------------------------------------------------------------------------------------------------
| current_epoch: 1.00000 | arithmetic_mean_loss: 0.11480 | harmonic_mean_loss: 1.23190 | prediction_mean_spearman: 0.61170 | entropy_spearman: 1.00000 |
--------------------------------------------------------------------------------------------------------------------------------------------------------



Validation: |                                                                                                 …


--------------------------------------------------------------------------------------------------------------------------------------------------------
| current_epoch: 2.00000 | arithmetic_mean_loss: 0.09895 | harmonic_mean_loss: 0.85955 | prediction_mean_spearman: 0.66365 | entropy_spearman: 1.00000 |
--------------------------------------------------------------------------------------------------------------------------------------------------------



## Reload best epoch and save

In [10]:
import tempfile
import re
import sys
import os

def set_best(my_model, callbacks):
    """
    Set the best model checkpoint for the provided model.

    This function sets the state of the provided model to the state of the best checkpoint,
    as determined by the `ModelCheckpoint` callback.

    Args:
        my_model (nn.Module): The model to be updated.
        callbacks (dict): Dictionary of callbacks, including 'model_checkpoint'.

    Returns:
        nn.Module: The updated model.
    """
    with tempfile.TemporaryDirectory() as tmpdirname:
        try:
            best_path = callbacks['model_checkpoint'].best_model_path
            get_epoch = re.search('epoch=(\d*)', best_path).group(1)
            if 'gs://' in best_path:
                subprocess.call(['gsutil','cp',best_path,tmpdirname])
                best_path = os.path.join( tmpdirname, os.path.basename(best_path) )
            print(f'Best model stashed at: {best_path}', file=sys.stderr)
            print(f'Exists: {os.path.isfile(best_path)}', file=sys.stderr)
            ckpt = torch.load( best_path )
            my_model.load_state_dict( ckpt['state_dict'] )
            print(f'Setting model from epoch: {get_epoch}', file=sys.stderr)
        except KeyError:
            print('Setting most recent model', file=sys.stderr)
    return my_model

graph = set_best(graph, {'model_checkpoint': checkpoint_callback})

Best model stashed at: /home/ubuntu/boda2/tutorials/lightning_logs/version_13/checkpoints/epoch=4-step=2820.ckpt
Exists: True
Setting model from epoch: 4


In [11]:
torch.save(graph.model.state_dict(), 'example_new_model.pt')

## load the save

In [12]:
new_model = model_module(
    n_outputs=2, 
    n_linear_layers=1, linear_channels=1000,
    linear_activation='ReLU', linear_dropout_p=0.12, 
    n_branched_layers=3, branched_channels=140, 
    branched_activation='ReLU', branched_dropout_p=0.56, 
    loss_criterion='L1KLmixed', loss_args={'beta':5.0}
)

new_model.load_state_dict(torch.load('example_new_model.pt'))
new_model.eval()
new_model.cuda()

BassetBranched(
  (pad1): ConstantPad1d(padding=(9, 9), value=0.0)
  (conv1): Conv1dNorm(
    (conv): Conv1d(4, 300, kernel_size=(19,), stride=(1,))
    (bn_layer): BatchNorm1d(300, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (pad2): ConstantPad1d(padding=(5, 5), value=0.0)
  (conv2): Conv1dNorm(
    (conv): Conv1d(300, 200, kernel_size=(11,), stride=(1,))
    (bn_layer): BatchNorm1d(200, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (pad3): ConstantPad1d(padding=(3, 3), value=0.0)
  (conv3): Conv1dNorm(
    (conv): Conv1d(200, 200, kernel_size=(7,), stride=(1,))
    (bn_layer): BatchNorm1d(200, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (pad4): ConstantPad1d(padding=(1, 1), value=0.0)
  (maxpool_3): MaxPool1d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
  (maxpool_4): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
  (linear1): LinearNorm(
    (linear): Linear(in

In [13]:
new_model( torch.randn(5,4,600).cuda() )

tensor([[30.3106, 58.3631],
        [32.2132, 56.4480],
        [30.0093, 56.7785],
        [26.7668, 45.2139],
        [25.1147, 43.6605]], device='cuda:0', grad_fn=<PermuteBackward0>)